# nb_05 — Interactive scatter + light-curve explorer

Goal (see `docs/SPEC_V01.md`, rough plan step 6): a reusable function — click a point in any
scatter plot (CMD, period-amplitude, ...), see that object's light curve on the right, with
toggles to fold on a period and to show/hide flux errors. Implemented once in
`src/visualization/lc_explorer.py` (`interactive_scatter_lc`), not copy-pasted per notebook —
this notebook is a demo of it against `dia_object_lc_hq` (nb_01's base HQ sample with nb_02's
stats, nb_03's magnitudes, and nb_04's periods all merged into it in place).

**Tech stack: `holoviews` + `bokeh` + `panel`, not `plotly`.** nb-v01 built this on
`plotly.graph_objects.FigureWidget` + `ipywidgets`, which needed `anywidget` installed
separately (not in the RSP kernel by default) and, once, a browser reload before the widget's
frontend model registered. `holoviews`/`bokeh`/`panel` all ship in the RSP `lsst-scipipe`
kernel already — matching the stack RSP's own interactive-plot tutorials use
(`notebooks/tutorials/DP2/300_Science_demos/312_Interactive_plots`) — so there's nothing extra
to install, and one less thing to go wrong for workshop attendees who've already seen that
stack in an earlier tutorial.

**`datapaths` is optional here, unlike in nb_01-04.** Those notebooks read/write several
artifacts and need `datapaths` to resolve and register them; this one only *reads* the single,
already-built `dia_object_lc_hq` collection, which workshop attendees won't be regenerating —
the setup cell below falls back to that collection's known shared path directly if `datapaths`
isn't installed/configured, so there's nothing to install just to run this notebook.

**HQ sample is too large to materialize whole for a live click demo.** nb-v01's version pulled
its entire subset (7,036 objects, ~90 MB) into memory once, since `interactive_scatter_lc`'s
`lc_df` has to already be materialized for a click to feel instant. The HQ sample is ~399k
objects — section 1 below picks a slice (one partition or a cone search, same
`select_slice` helper as nb_02-04) instead of the whole thing.


In [23]:
import sys
from pathlib import Path

# src/ isn't pip-installed on RSP (pyproject.toml is pinned to Python 3.14, newer than RSP's
# 3.13.9 kernel — see README) — import it straight from the repo checkout instead.
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

import lsdb
from dataio import select_slice
from visualization import interactive_scatter_lc, plot_lightcurve

try:
    # Only needed to *resolve* dia_object_lc_hq's path from configs/artifacts_registry.yaml +
    # a local configs/roots.local.yaml — not needed at all for workshop attendees, who are only
    # reading the already-built collection below, not running nb_01-04 or registering their own
    # artifacts. Falls back to the collection's known path directly when datapaths isn't
    # installed/configured, so installing it isn't a prerequisite for this notebook.
    from datapaths import Datapaths

    dp = Datapaths()
    hq_path = dp["dp2_subset"] / "dia_object_lc_hq"
except Exception as exc:
    # Every RSP user's home has `share/trieste` mounted to the same shared location (see
    # README's roots.local.yaml example: dp2_subset -> ~/share/trieste) — read the collection
    # straight from there instead of through datapaths.
    print(f"datapaths unavailable ({type(exc).__name__}: {exc}) — falling back to the shared path directly.")
    hq_path = Path.home() / "share" / "trieste" / "dia_object_lc_hq"

hq_cat = lsdb.open_catalog(hq_path)
print(f"reading {hq_path}")
print(hq_cat.npartitions, "partitions")
print(sorted(hq_cat.columns))

reading /home/shrra-ung/share/trieste/dia_object_lc_hq
4445 partitions
['best_period_days', 'best_period_power', 'dec', 'diaObjectForcedSource', 'diaObjectId', 'diaSource', 'duration_days', 'g_amp_p90p10', 'g_mag_median', 'g_period_days', 'g_period_power', 'i_amp_p90p10', 'i_mag_median', 'i_period_days', 'i_period_power', 'max_reliability', 'median_cadence_gap_days', 'nDiaSources', 'periodogram_peaks', 'r_amp_p90p10', 'r_mag_median', 'r_period_days', 'r_period_power', 'ra', 'tract', 'u_amp_p90p10', 'u_mag_median', 'u_period_days', 'u_period_power', 'y_amp_p90p10', 'y_mag_median', 'y_period_days', 'y_period_power', 'z_amp_p90p10', 'z_mag_median', 'z_period_days', 'z_period_power']


## 1. Pick a slice of the HQ sample to explore

Same `select_slice` pattern as nb_02-04, but here the slice *is* the working dataset for the
rest of the notebook, not just something to glance at — `interactive_scatter_lc`'s `lc_df`
needs to already be materialized (fetching a light curve per click has to be fast, no lazy
per-click `.compute()`), and the whole ~399k-object HQ sample is too big to pull into memory
for that. One partition or one cone search gives a few dozen to a few hundred objects — small
enough to hold in memory and reuse as both the light-curve source and the scatter-plot source
below.
</cell id="3112e303">

In [24]:
SLICE_MODE = "partition"  # "partition" or "cone_search"
PARTITION_INDEX = 200
CONE_RA, CONE_DEC, CONE_RADIUS_ARCSEC = 150.0, 2.0, 1800  # ~0.5 deg

slice_cat = select_slice(
    hq_cat,
    mode=SLICE_MODE,
    partition_index=PARTITION_INDEX,
    ra=CONE_RA,
    dec=CONE_DEC,
    radius_arcsec=CONE_RADIUS_ARCSEC,
)
full_df = slice_cat.compute()
print(f"{SLICE_MODE}: {full_df.shape}")

Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

partition: (42, 37)


In [25]:
hq_cat.head()

Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

dec         diaObjectId  nDiaSources          ra  \
_healpix_29                                                                  
1807629683194439178 -8.527954  776659829488877572          123  212.400233   
1807630566727558341 -8.531481  776659898208354321          102  212.282207   
1807631025111336059  -8.49711  776660585403121701          105  212.290012   
1807631095368997469 -8.493643  776653094980157452          124  212.246302   
1807631491086875027 -8.485029  776660516683644949          102  212.384883   

                     tract  duration_days  median_cadence_gap_days  \
_healpix_29                                                          
1807629683194439178   8161       4.938721                 0.000975   
1807630566727558341   8161       6.845205                 0.001465   
1807631025111336059   8161       6.845205                 0.001009   
1807631095368997469   8160       6.845695                 0.001009   
1807631491086875027   8161       6.845695                 0.001447   

                     u_amp_p90p10  g_amp_p90p10  r_amp_p90p10  ...  \
_healpix_29                                                    ...   
1807629683194439178          <NA>      0.564366      0.226382  ...   
1807630566727558341          <NA>      0.019428      0.075232  ...   
1807631025111336059          <NA>      0.128798      0.027671  ...   
1807631095368997469          <NA>      0.017521      0.034527  ...   
1807631491086875027          <NA>      0.237222      0.130561  ...   

                     i_period_power  z_period_days  z_period_power  \
_healpix_29                                                          
1807629683194439178        0.588659           <NA>            <NA>   
1807630566727558341        0.213853           <NA>            <NA>   
1807631025111336059        0.093674           <NA>            <NA>   
1807631095368997469        0.441977           <NA>            <NA>   
1807631491086875027        0.284475           <NA>            <NA>   

                     y_period_days  y_period_power  best_period_days  \
_healpix_29                                                            
1807629683194439178           <NA>            <NA>          0.570381   
1807630566727558341           <NA>            <NA>          0.030433   
1807631025111336059           <NA>            <NA>          0.055091   
1807631095368997469           <NA>            <NA>          0.029607   
1807631491086875027           <NA>            <NA>          0.072751   

                     best_period_power  \
_healpix_29                              
1807629683194439178            0.71164   
1807630566727558341           0.213853   
1807631025111336059           0.141946   
1807631095368997469           0.441977   
1807631491086875027           0.307382   

                                                 diaObjectForcedSource  \
_healpix_29                                                              
1807629683194439178  [{band: 'g', coord_dec: -8.527954, coord_ra: 2...   
1807630566727558341  [{band: 'g', coord_dec: -8.531481, coord_ra: 2...   
1807631025111336059  [{band: 'g', coord_dec: -8.49711, coord_ra: 21...   
1807631095368997469  [{band: 'g', coord_dec: -8.493643, coord_ra: 2...   
1807631491086875027  [{band: 'g', coord_dec: -8.485029, coord_ra: 2...   

                                                             diaSource  \
_healpix_29                                                              
1807629683194439178  [{band: 'g', centroid_flag: False, dec: -8.527...   
1807630566727558341  [{band: 'g', centroid_flag: False, dec: -8.531...   
1807631025111336059  [{band: 'g', centroid_flag: False, dec: -8.497...   
1807631095368997469  [{band: 'g', centroid_flag: False, dec: -8.493...   
1807631491086875027  [{band: 'g', centroid_flag: False, dec: -8.485...   

                                                     periodogram_peaks  
_healpix_29                                                             
1807629683194439178  [

In [26]:
slice_cat.columns

Index(['dec', 'diaObjectId', 'nDiaSources', 'ra', 'tract', 'duration_days',
       'median_cadence_gap_days', 'u_amp_p90p10', 'g_amp_p90p10',
       'r_amp_p90p10', 'i_amp_p90p10', 'z_amp_p90p10', 'y_amp_p90p10',
       'u_mag_median', 'g_mag_median', 'r_mag_median', 'i_mag_median',
       'z_mag_median', 'y_mag_median', 'max_reliability', 'u_period_days',
       'u_period_power', 'g_period_days', 'g_period_power', 'r_period_days',
       'r_period_power', 'i_period_days', 'i_period_power', 'z_period_days',
       'z_period_power', 'y_period_days', 'y_period_power', 'best_period_days',
       'best_period_power', 'diaObjectForcedSource', 'diaSource',
       'periodogram_peaks'],
      dtype='object')

## 2. Demo: color-magnitude diagram (nb_03)

`g-r` vs `r`, colored by `max_reliability` (nb_03's per-object max real/bogus score), folded
on `best_period_days` (nb_04's single-band period) when available. Click a point on the left
to load its light curve on the right; toggle folded vs. unfolded and flux errors on/off.
</cell id="c2bbe626">

In [27]:
full_df['diaObjectForcedSource'].columns

['band',
 'coord_dec',
 'coord_ra',
 'diff_PixelFlags_nodataCenter',
 'invalidPsfFlag',
 'midpointMjdTai',
 'pixelFlags_bad',
 'pixelFlags_cr',
 'pixelFlags_crCenter',
 'pixelFlags_edge',
 'pixelFlags_interpolated',
 'pixelFlags_interpolatedCenter',
 'pixelFlags_nodata',
 'pixelFlags_saturated',
 'pixelFlags_saturatedCenter',
 'pixelFlags_suspect',
 'pixelFlags_suspectCenter',
 'psfDiffFlux',
 'psfDiffFlux_flag',
 'psfDiffFluxErr',
 'psfFlux',
 'psfFlux_flag',
 'psfFluxErr',
 'psfMag',
 'psfMagErr',
 'visit']

In [28]:
cmd_df = full_df.assign(gr=full_df["g_mag_median"] - full_df["r_mag_median"]).dropna(subset=["gr", "r_mag_median"])
print(cmd_df.shape)

(42, 38)


In [29]:
interactive_scatter_lc(
    scatter_df=cmd_df,
    x_col="gr",
    y_col="r_mag_median",
    lc_df=full_df,
    color_col="max_reliability",
    period_col="best_period_days",
    scatter_title="CMD: g-r vs r",
    mag_col="psfDiffFlux", magerr_col="psfDiffFluxErr", nested_col="diaObjectForcedSource"
)

Row
    [0] HoloViews(Points, height=460, sizing_mode='fixed', width=460)
    [1] Column
        [0] HTML(str)
        [1] HTML(str)
        [2] Row
            [0] RadioButtonGroup(label='fold', name='fold', options={'Folded': True, ...}, value=True)
            [1] Checkbox(label='show flux errors', name='show flux errors', value=True)
        [3] HoloViews(DynamicMap, height=460, sizing_mode='fixed', width=560)

### 2b. Failsafe: manual light-curve plotting

The widget above shows a **"selected id" line** right under the scatter/light-curve panels,
updated straight off the click — independent of the light-curve panel itself. If the light
curve on the right ever stops following your clicks (e.g. a stale `bokeh`/`panel` browser comm
that needs a page reload — not something we want to depend on mid-workshop), that line still
tells you which `diaObjectId` you clicked.

Read that id and pass it to `plot_lightcurve` — a plain function, no click/stream/widget
involved — to get the same plot in a separate cell. `fold` is a parameter, not a toggle: pass
the object's period and `fold=True`/`False` explicitly.


In [30]:
# Manual fallback: paste the diaObjectId shown by the widget's "selected id" line here.
SELECTED_OBJ_ID = cmd_df["diaObjectId"].iloc[20]  # placeholder — replace with the id you clicked

selected_row = cmd_df.loc[cmd_df["diaObjectId"] == SELECTED_OBJ_ID].iloc[0]
period = selected_row["best_period_days"] if pd.notna(selected_row["best_period_days"]) else None

plot_lightcurve(
    full_df,
    obj_id=SELECTED_OBJ_ID,
    mag_col="psfDiffFlux", magerr_col="psfDiffFluxErr", nested_col="diaObjectForcedSource",
    period=period,
    fold=False,  # flip to True to fold on `period`
)

:NdOverlay   [band]
   :Overlay
      .ErrorBars.I :ErrorBars   [x]   (y,err)
      .Scatter.I   :Scatter   [x]   (y)

In [31]:
# Manual fallback: paste the diaObjectId shown by the widget's "selected id" line here.
SELECTED_OBJ_ID = cmd_df["diaObjectId"].iloc[15]  # placeholder — replace with the id you clicked

selected_row = cmd_df.loc[cmd_df["diaObjectId"] == SELECTED_OBJ_ID].iloc[0]
period = selected_row["best_period_days"] if pd.notna(selected_row["best_period_days"]) else None

plot_lightcurve(
    full_df,
    obj_id=SELECTED_OBJ_ID,
    mag_col="psfMag", magerr_col="psfMagErr", nested_col="diaObjectForcedSource",
    period=period,bands='gri',
    fold=False,  # flip to True to fold on `period`
)

:NdOverlay   [band]
   :Overlay
      .ErrorBars.I :ErrorBars   [x]   (y,err)
      .Scatter.I   :Scatter   [x]   (y)

## 3. Demo: period-amplitude diagram (nb_04 x nb_02)

`multiband_period_days` (log-scaled, nb_04's higher-coverage period) vs. `r_amp_p90p10`
(nb_02's robust amplitude), colored by `duration_days` and folded on the same
`multiband_period_days`. This is exactly the combination the spec's rough plan step 5 asks for
as a static plot — here it's interactive instead, in the same function used for the CMD above.
</cell id="ffdb54e6">

In [32]:
full_df.columns

Index(['dec', 'diaObjectId', 'nDiaSources', 'ra', 'tract', 'duration_days',
       'median_cadence_gap_days', 'u_amp_p90p10', 'g_amp_p90p10',
       'r_amp_p90p10', 'i_amp_p90p10', 'z_amp_p90p10', 'y_amp_p90p10',
       'u_mag_median', 'g_mag_median', 'r_mag_median', 'i_mag_median',
       'z_mag_median', 'y_mag_median', 'max_reliability', 'u_period_days',
       'u_period_power', 'g_period_days', 'g_period_power', 'r_period_days',
       'r_period_power', 'i_period_days', 'i_period_power', 'z_period_days',
       'z_period_power', 'y_period_days', 'y_period_power', 'best_period_days',
       'best_period_power', 'diaObjectForcedSource', 'diaSource',
       'periodogram_peaks'],
      dtype='object')

In [33]:
pa_df = full_df.dropna(subset=["r_period_days", "r_amp_p90p10"])
print(pa_df.shape)

interactive_scatter_lc(
    scatter_df=pa_df,
    x_col="r_period_days",
    y_col="r_amp_p90p10",
    lc_df=full_df,
    color_col="duration_days",
    period_col="r_period_days",
    scatter_title="period-amplitude: r_period_days vs r-band amplitude",
    mag_col="psfMag", magerr_col="psfMagErr", nested_col="diaObjectForcedSource",
    x_log=True,
)

(39, 37)


Row
    [0] HoloViews(Points, height=460, sizing_mode='fixed', width=460)
    [1] Column
        [0] HTML(str)
        [1] HTML(str)
        [2] Row
            [0] RadioButtonGroup(label='fold', name='fold', options={'Folded': True, ...}, value=True)
            [1] Checkbox(label='show flux errors', name='show flux errors', value=True)
        [3] HoloViews(DynamicMap, height=460, sizing_mode='fixed', width=560)

In [34]:
# Manual fallback: paste the diaObjectId shown by the widget's "selected id" line here.
SELECTED_OBJ_ID = 788013730154676251  # placeholder — replace with the id you clicked

selected_row = full_df.loc[full_df["diaObjectId"] == SELECTED_OBJ_ID].iloc[0]
period = selected_row["best_period_days"] if pd.notna(selected_row["best_period_days"]) else None

plot_lightcurve(
    full_df,
    obj_id=SELECTED_OBJ_ID,
    mag_col="psfMag", magerr_col="psfMagErr", nested_col="diaObjectForcedSource",
    period=period,bands='gri',
    fold=False,  # flip to True to fold on `period`
)

:NdOverlay   [band]
   :Overlay
      .ErrorBars.I :ErrorBars   [x]   (y,err)
      .Scatter.I   :Scatter   [x]   (y)

## Next

`src/visualization/lc_explorer.py`'s `interactive_scatter_lc` is the reusable piece; this
notebook is just two example calls against real HQ-sample output. Open questions, not resolved
here:

- **Not tested outside JupyterLab on RSP.** The spec explicitly flags Jupyter-vs-VSCode/IDE
  differences for interactive widgets — unverified either way here.
- **Categorical `color_col` legend — resolved by the bokeh/holoviews rewrite.** The old
  plotly version couldn't show a per-category legend on a single trace (category rode along in
  hover text instead); `holoviews`'s native categorical coloring shows a real legend, no
  workaround needed. Verified against a categorical column during this rewrite (see
  `docs/changelog.md`), though neither demo above happens to use one (HQ's `max_reliability`/
  `duration_days` are both numeric).
- **`lc_df` must already be materialized** — no support for handing it a lazy `lsdb.Catalog`
  and computing per click. Section 1 now picks a slice of the HQ sample specifically because of
  this; would need rethinking (streaming fetch per click?) before pointing this at the full
  ~399k-object sample directly.
- **Doesn't add its own quality filtering.** All of nb_04's caveats about `*_period_power` not
  being a calibrated false-alarm probability still apply when browsing by period here — clicking
  a high-power point doesn't mean the fold is real.
- **Single light-curve panel, not one subplot per band** — matches the spec's wording ("the LC
  plotting panel", singular), but multi-band light curves with very different flux scales can
  be hard to read overlaid; worth revisiting if that turns out to matter in practice.
</cell id="c7abba23">